In [1]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import numpy as np
import pandas as pd


In [32]:
#### Load the ANn trsine model,scalr,pickle,onehot
model=load_model('model.h5')

## load the encoder and scaler
with open('onehot_encoder_geography.pkl','rb') as file:
    Onehot_encoder_geography=pickle.load(file)

with open('scaler.pkl','rb') as file:
    scaler=pickle.load(file)   

with open('label_encoder_gender.pkl','rb') as file:
    label_encoder_gender=pickle.load(file)     

In [33]:
##eXAMPLE INPUT DATA
input_data={
    'CreditScore':600,
    'Geography':'France',
    'Gender':'Male',
    'Age':40,
    'Tenure':3,
    'Balance':60000,
    'NumOfProducts':2,
    'HasCrCard':1,
    'IsActiveMember':1,
    'EstimatedSalary':50000
}

In [34]:
##### Onehot_encoder_geography is the pickle file for the onehot encoder used to encode the 'Geography' column in the training data. We will use it to encode the 'Geography' value in the input data before making predictions with the ANN model. The code snippet above demonstrates how to use the loaded onehot encoder to transform the 'Geography' value from the input data into a one-hot encoded format, which can then be used as input for the ANN model.   


geo_encoded=Onehot_encoder_geography.transform([[input_data['Geography']]]).toarray()
geo_encoded_df=pd.DataFrame(geo_encoded,columns=Onehot_encoder_geography.get_feature_names_out(['Geography']))
geo_encoded_df

c:\Users\Lenovo\Desktop\PROJECT ANN\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [35]:
input_df=pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [36]:
### Combine one_hot encoded column with input data 
input_data=pd.concat([input_df.reset_index(drop=True),geo_encoded_df],axis=1)
input_data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,France,Male,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [37]:
input_df['Gender']=label_encoder_gender.transform(input_df['Gender'])

In [38]:
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,40,3,60000,2,1,1,50000


In [42]:
###concatination one hot encoded 
## input_df=pd.concat([input_df.drop("Geography",axis=1),geo_encoded_df],axis=1)
input_df = pd.concat([input_df.drop('Geography', axis=1, errors='ignore'), geo_encoded_df], axis=1)

input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0,1.0,0.0,0.0


In [ ]:
# Remove duplicate columns
input_df = input_df.loc[:, ~input_df.columns.duplicated()]

# Reorder to match training
expected_columns = scaler.feature_names_in_
input_df = input_df[expected_columns]

# Now scale
input_scaled = scaler.transform(input_df)


In [47]:
input_scaled


array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [49]:
##PREDICTION CHUR
predicition=model.predict(input_scaled)
predicition

1/1 [==============================] - 0s 89ms/step


array([[0.03150954]], dtype=float32)

In [50]:
predicition_proba=predicition[0][0]
predicition_proba

0.03150954

In [51]:
if predicition_proba > 0.5:
    print("The customer is likely to churn.")               
else:   
    print("The customer is unlikely to churn.")
    

The customer is unlikely to churn.
